<a href="https://colab.research.google.com/github/xinccojp/create-aivis-model-colab/blob/main/colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#@markdown # このノートブックについて

#@markdown このノートブックは[litagin02/Style-Bert-VITS2リポジトリにあるGoogle Colabノートブック](https://colab.research.google.com/github/litagin02/Style-Bert-VITS2/blob/master/colab.ipynb)をベースに、声の学習を行なうためだけに簡略化させたノートブックとなります。

#@markdown 音声ファイルと学習結果はマウントしたGoogle Drive上（`MyDrive/Style-Bert-VITS2`）に保存されるので、ランタイムが切れても消えません。

#@markdown プログラムが分かる方・より使いこなしていきたい方はベースとさせてもらった [litagin02/Style-Bert-VITS2](https://github.com/litagin02/Style-Bert-VITS2) をご覧ください。

#@markdown ---
#@markdown ## 実行順
#@markdown 0. Google Driveをマウント
#@markdown 1. セットアップ（初回はランタイム再起動後にもう一度実行）
#@markdown 2. 学習用モデルのダウンロード
#@markdown 3. 互換性パッチ（ランタイム再起動のたびに実行）
#@markdown 4. 音声ファイルの変換（`inputs` にwavを置いてから）
#@markdown 5. 学習
#@markdown 6. Web UIで試す / ONNX変換

In [ ]:
#@markdown # 0. Google Driveをマウント
#@markdown 音声ファイルの置き場と学習結果の保存先としてGoogle Driveを使います。
#@markdown 実行すると認証を求められるので、許可してください。
from google.colab import drive

drive.mount("/content/drive")

# Drive上の作業フォルダ（このノートブック全体で使うパス）
drive_root = "/content/drive/MyDrive/Style-Bert-VITS2"

# 学習に必要なファイルや途中経過が保存されるディレクトリ
dataset_root = f"{drive_root}/Data"
# 学習結果（音声合成に必要なファイルたち）が保存されるディレクトリ
assets_root = f"{drive_root}/model_assets"
# 元となる音声ファイル（wav形式）を入れるディレクトリ
input_dir = f"{drive_root}/inputs"

!mkdir -p "{dataset_root}"
!mkdir -p "{assets_root}"
!mkdir -p "{input_dir}"

print("マウント完了。以下のフォルダにwavを入れてください:")
print(input_dir)

In [ ]:
#@markdown # 1. セットアップ
#@markdown 依存パッケージのインストールを行ないます。
#@markdown 初回実行時はインストール後にランタイムが再起動されるので、再起動後にもう一度このセルを実行してください。
import os

if not os.path.exists("/content/Style-Bert-VITS2"):
    !git clone https://github.com/litagin02/Style-Bert-VITS2.git

%cd /content/Style-Bert-VITS2/

try:
    import loguru
    import numpy
    setup_done = (numpy.__version__ == "1.26.4")
except ImportError:
    setup_done = False

if not setup_done:
    print("初回セットアップを開始します...")

    # Colab 既定の pip constraint は nltk と numpy のバージョンを固定していて
    # requirements-colab.txt と衝突するため、空の constraint に差し替える
    open("/content/constraints-sbv2.txt", "w").close()
    os.environ["PIP_CONSTRAINT"] = "/content/constraints-sbv2.txt"

    # nltk<=3.8.1 の指定は Colab 同梱の nltk 3.9.1 と衝突するので上限を外す
    !sed -i 's/^nltk.*/nltk/' requirements-colab.txt
    # pyannote.audio 4.x は numpy>=2 を要求し requirements の numpy<2 と衝突するので 3 系に固定
    !sed -i 's/^pyannote\.audio.*/pyannote.audio==3.3.2/' requirements-colab.txt

    !pip install -r requirements-colab.txt
    !pip install --no-cache-dir "numpy==1.26.4"

    print("環境を反映させるため再起動します。再起動後、もう一度このセルを実行してください。")
    os.kill(os.getpid(), 9)

# Drive上の作業フォルダ（「0. Google Driveをマウント」と同じパス）
drive_root = "/content/drive/MyDrive/Style-Bert-VITS2"
dataset_root = f"{drive_root}/Data"
assets_root = f"{drive_root}/model_assets"
input_dir = f"{drive_root}/inputs"

!mkdir -p "{dataset_root}"
!mkdir -p "{assets_root}"
!mkdir -p "{input_dir}"

import importlib.util
for m in ["loguru", "librosa", "cmudict", "g2p_en", "pyannote.audio", "pyopenjtalk"]:
    print(m, "OK" if importlib.util.find_spec(m) else "MISSING")

print("準備が整いました。")

In [ ]:
#@markdown # 2. 学習用モデルのダウンロード
#@markdown BERTモデルや事前学習モデルをダウンロードします。数分かかります。
%cd /content/Style-Bert-VITS2

import numpy
print("numpy", numpy.__version__)   # 1.26.4 であること

!python initialize.py

In [ ]:
#@markdown # 3. 互換性パッチ
#@markdown Colab の torch / torchaudio / huggingface_hub が新しく、pyannote.audio 3.x と噛み合わないため
#@markdown 不足しているAPIを補います。ランタイムを再起動したら再度実行してください（適用済みならスキップされます）。
import huggingface_hub
import torch
import torchaudio


def append_patch(target_file, patch_code, marker="互換性パッチ"):
    with open(target_file, "r", encoding="utf-8") as f:
        content = f.read()
    if marker in content:
        print(f"すでに適用済み: {target_file}")
        return
    with open(target_file, "a", encoding="utf-8") as f:
        f.write(patch_code)
    print(f"パッチ適用完了: {target_file}")


# torchaudio: 新しいバージョンで削除された API をダミーで復元する。
# モジュール末尾に追記されるため、モジュール自身の名前空間に直接定義する
# （`torchaudio.xxx = ...` と書くと NameError になる）。
append_patch(torchaudio.__file__, '''

# --- 互換性パッチ: 新しいtorchaudioで削除されたAPIをダミーで復元 ---
if "list_audio_backends" not in globals():
    def list_audio_backends():
        return ["soundfile"]

if "AudioMetaData" not in globals():
    class AudioMetaData:
        def __init__(self, sample_rate=0, num_frames=0, num_channels=0, bits_per_sample=0, encoding=""):
            self.sample_rate = sample_rate
            self.num_frames = num_frames
            self.num_channels = num_channels
            self.bits_per_sample = bits_per_sample
            self.encoding = encoding
# --- パッチここまで ---
''')

# huggingface_hub: 廃止された use_auth_token 引数を token に読み替える
append_patch(huggingface_hub.__file__, '''

# --- 互換性パッチ: use_auth_token -> token の引数名変換 ---
from huggingface_hub.file_download import hf_hub_download as _original_hf_hub_download

def _patched_hf_hub_download(*args, **kwargs):
    if "use_auth_token" in kwargs:
        kwargs["token"] = kwargs.pop("use_auth_token")
    return _original_hf_hub_download(*args, **kwargs)

import huggingface_hub.file_download as _hf_file_download_module
_hf_file_download_module.hf_hub_download = _patched_hf_hub_download

hf_hub_download = _patched_hf_hub_download
# --- パッチここまで ---
''')

# torch.load: PyTorch 2.6 以降の weights_only=True 既定を無効化する。
# 公式配布の pyannote チェックポイントが読めなくなるため。
append_patch(torch.__file__, '''

# --- 互換性パッチ: torch.load のデフォルトを weights_only=False に戻す ---
# (信頼できる公式モデル読み込みのため。PyTorch 2.6以降のデフォルト変更に対応)
_original_torch_load = torch.load

def _patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return _original_torch_load(*args, **kwargs)

torch.load = _patched_torch_load
# --- パッチここまで ---
''')

# bert_models.py: transformers 5.x で torch_dtype が廃止されたため dtype を明示する。
# `import torch` が TYPE_CHECKING の下にしかないので実行時用の import も追加する。
bert_models_path = "/content/Style-Bert-VITS2/style_bert_vits2/nlp/bert_models.py"

with open(bert_models_path, "r", encoding="utf-8") as f:
    content = f.read()

if "dtype=torch.float32" in content:
    print(f"すでに適用済み: {bert_models_path}")
else:
    content = content.replace(
        "from __future__ import annotations\n",
        "from __future__ import annotations\n\nimport torch\n",
        1,
    )
    content = content.replace(
        """                cache_dir=cache_dir,
                revision=revision,
            ),""",
        """                cache_dir=cache_dir,
                revision=revision,
                dtype=torch.float32,
            ),""",
    )
    # トークナイザー側の from_pretrained と区別するため device_map の行込みで置換する
    content = content.replace(
        """            pretrained_model_name_or_path,
            device_map=device_map,
            cache_dir=cache_dir,
            revision=revision,
        )""",
        """            pretrained_model_name_or_path,
            device_map=device_map,
            cache_dir=cache_dir,
            revision=revision,
            dtype=torch.float32,
        )""",
    )
    with open(bert_models_path, "w", encoding="utf-8") as f:
        f.write(content)
    print(f"パッチ適用完了: {bert_models_path}")

In [ ]:
#@markdown ## パッチの動作確認
#@markdown 3つとも OK が出れば次に進めます。
!python -c "import torchaudio; print('torchaudio OK', torchaudio.list_audio_backends(), torchaudio.AudioMetaData)"
!python -c "from pyannote.audio import Inference, Model; print('pyannote OK')"
!python -c "import torch; import sys; sys.path.insert(0, '/content/Style-Bert-VITS2'); from style_bert_vits2.nlp import bert_models; print('bert_models OK')"

In [ ]:
#@markdown # 4. 音声ファイルの変換
#@markdown Drive上の `MyDrive/Style-Bert-VITS2/inputs` に wav を置いてから実行してください。
#@markdown 44100Hz / モノラル / 16bit PCM に揃えて、元のファイルと差し替えます。
import glob
import os
import shutil
import subprocess

input_dir = "/content/drive/MyDrive/Style-Bert-VITS2/inputs"
# 変換作業はDriveではなくローカルディスク上で行なう（Driveへの書き込みは遅い）
converted_dir = "/content/converted_inputs"
os.makedirs(converted_dir, exist_ok=True)

wav_files = glob.glob(os.path.join(input_dir, "*.wav"))
print(f"{len(wav_files)} 件のファイルを変換します")

converted = []
for f in wav_files:
    basename = os.path.basename(f)
    out_path = os.path.join(converted_dir, basename)
    cmd = [
        "ffmpeg", "-y", "-i", f,
        "-ar", "44100",      # サンプリングレートを44100Hzに統一
        "-ac", "1",          # モノラルに統一
        "-c:a", "pcm_s16le", # 16bit PCMに変換
        out_path,
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    if result.returncode != 0:
        print(f"失敗: {basename}")
        print(result.stderr[-500:])
    else:
        converted.append(out_path)
        print(f"成功: {basename}")

# 変換に成功したものだけ元のファイルと差し替える
for f in converted:
    shutil.copy(f, input_dir)

print("入れ替え完了")
!ls -la "{input_dir}"

In [ ]:
#@markdown # 5. 学習
%cd /content/Style-Bert-VITS2

# Drive上の作業フォルダ（「0. Google Driveをマウント」と同じパス）
drive_root = "/content/drive/MyDrive/Style-Bert-VITS2"

# 学習に必要なファイルや途中経過が保存されるディレクトリ
dataset_root = f"{drive_root}/Data"

# 学習結果（音声合成に必要なファイルたち）が保存されるディレクトリ
assets_root = f"{drive_root}/model_assets"

# 元となる音声ファイル（wav形式）を入れるディレクトリ
input_dir = f"{drive_root}/inputs"

import yaml

with open("configs/paths.yml", "w", encoding="utf-8") as f:
    yaml.dump({"dataset_root": dataset_root, "assets_root": assets_root}, f)

#@markdown ---
#@markdown ## 作成するモデル関連の内容を入力
#@markdown ※デフォルトのままでも問題ありません
# モデル名（話者名）を入力
model_name = "your_model_name" #@param {type:"string"}

# こういうふうに書き起こして欲しいという例文（句読点の入れ方・笑い方や固有名詞等）
initial_prompt = "こんにちは。元気、ですかー？ふふっ、私は……ちゃんと元気だよ！" #@param {type:"string"}

#@markdown ---
#@markdown ## JP-Extraを有効化する
#@markdown JP-Extra （日本語特化版）を有効化すると、日本語の能力が向上する代わりに英語と中国語は使えなくなります。
use_jp_extra = True #@param {type: "boolean"}

#@markdown ---
#@markdown ## 学習のバッチサイズ。
#@markdown VRAMのはみ出具合に応じて調整してください。
batch_size = 4 #@param {type:"string"}

#@markdown ---
#@markdown ## 学習のエポック数
# 100で多すぎるほどかもしれませんが、もっと多くやると質が上がる可能性もあります
epochs = 100 #@param {type:"string"}

#@markdown ---
#@markdown ## 保存頻度
#@markdown 保存頻度。何ステップごとにモデルを保存するか。分からなければデフォルトのままで。
save_every_steps = 1000 #@param {type:"string"}

#@markdown ---
#@markdown 音声ファイルの音量を正規化するかどうか
normalize = False #@param {type:"string"}

#@markdown ---
#@markdown 音声ファイルの開始・終了にある無音区間を削除するかどうか
trim = False #@param {type:"string"}

#@markdown ---
#@markdown 読みのエラーが出た場合にどうするか。

#@markdown ・"raise"ならテキスト前処理が終わったら中断

#@markdown ・"skip"なら読めない行は学習に使わない

#@markdown ・"use"なら無理やり使う
yomi_error = "skip" #@param {type:"string"}

# 音声の切り出しと書き起こし
!python slice.py -i "{input_dir}" --model_name {model_name}
!python transcribe.py --model_name {model_name} --initial_prompt {initial_prompt} --use_hf_whisper --hf_repo_id openai/whisper-large-v3 --language ja

# 以降は学習に関する処理
from gradio_tabs.train import get_path, preprocess_all
from style_bert_vits2.nlp.japanese import pyopenjtalk_worker

pyopenjtalk_worker.initialize_worker()

preprocess_all(
    model_name=model_name,
    batch_size=batch_size,
    epochs=epochs,
    save_every_steps=save_every_steps,
    num_processes=2,
    normalize=normalize,
    trim=trim,
    freeze_EN_bert=False,
    freeze_JP_bert=False,
    freeze_ZH_bert=False,
    freeze_style=False,
    freeze_decoder=False,
    use_jp_extra=use_jp_extra,
    val_per_lang=0,
    log_interval=200,
    yomi_error=yomi_error,
)

paths = get_path(model_name)
dataset_path = str(paths.dataset_path)
config_path = str(paths.config_path)

with open("default_config.yml", "r", encoding="utf-8") as f:
    yml_data = yaml.safe_load(f)
yml_data["model_name"] = model_name
with open("config.yml", "w", encoding="utf-8") as f:
    yaml.dump(yml_data, f, allow_unicode=True)

if use_jp_extra:
  # 学習 （日本語特化版を「使う」場合）
  !python train_ms_jp_extra.py --config {config_path} --model {dataset_path} --assets_root "{assets_root}"
else:
  # 学習 （日本語特化版を「使わない」場合）
  !python train_ms.py --config {config_path} --model {dataset_path} --assets_root "{assets_root}"

In [ ]:
#@markdown # 6. Web UI起動
#@markdown Web UIを起動して学習させたモデルを試す事ができますが、Colabからだと音声の生成ができないかも？
!python app.py --share

In [ ]:
#@markdown # 7. ONNX変換
#@markdown 学習済みのモデルをONNXに変換します。

#@markdown 変換後は

#@markdown /content/drive/MyDrive/Style-Bert-VITS2/model_assets/{設定したモデル名}/{設定したモデル名}_e100_s300.onnx

#@markdown にモデルが出力されます
!git fetch -p
!git checkout -b dev origin/dev
!pip install -r requirements-colab.txt
!python convert_onnx.py --model "/content/drive/MyDrive/Style-Bert-VITS2/model_assets/{model_name}"